# Structural Crack Detection Using Deep Learning (SDNET2018)

## Project Overview

Structural cracks in concrete surfaces such as bridges, pavements, and walls are critical indicators of infrastructure degradation. Early detection is essential to ensure safety, reduce maintenance costs, and prevent catastrophic failures.

In this project, we develop a deep learning-based image classification system to detect cracks across different types of concrete surfaces using the SDNET2018 dataset.

The objective is to classify images into:

- Crack
- No Crack

This notebook follows a complete machine learning pipeline, including data loading, preprocessing, model training, evaluation, and comparison of multiple deep learning architectures.

## Research Question

Can deep learning models accurately detect cracks across multiple concrete surface types, including bridge decks, pavements, and walls?

This project also asks:

Which pretrained convolutional neural network performs best for structural crack detection when evaluated using recall, F1-score, and overall classification performance?

Because missing a crack can create serious safety risks, this project prioritizes recall for the crack class in addition to accuracy, precision, F1-score, and confusion matrix analysis.

## Data Source

This project uses the SDNET2018 dataset, a widely recognized dataset for structural defect detection in concrete surfaces.

The dataset includes images from three different types of surfaces:

- Bridge Decks (D)
- Pavements (P)
- Walls (W)

Each surface type contains two categories:

- Cracked (C*)
- Uncracked (U*)

The dataset structure allows us to combine all cracked images into a single "Crack" class and all uncracked images into a "No Crack" class, enabling a binary classification task.

SDNET2018 is considered a realistic dataset because it includes variations in lighting, texture, and surface conditions, making it suitable for evaluating deep learning models in real-world scenarios.

In [ ]:
# 1. Define Dataset Paths (SDNET2018)

import os

base_path = os.path.expanduser("~/Desktop/SDNET2018")

paths = {
    "CD": os.path.join(base_path, "D/CD"),
    "UD": os.path.join(base_path, "D/UD"),
    "CP": os.path.join(base_path, "P/CP"),
    "UP": os.path.join(base_path, "P/UP"),
    "CW": os.path.join(base_path, "W/CW"),
    "UW": os.path.join(base_path, "W/UW"),
}

# Check paths exist
for k, v in paths.items():
    print(k, "->", "Exists" if os.path.exists(v) else "Missing")

## Dataset Description

The SDNET2018 dataset consists of images of concrete surfaces categorized by surface type and crack presence.

Dataset structure:

- D (Bridge Decks)
  - CD: Cracked Deck
  - UD: Uncracked Deck

- P (Pavements)
  - CP: Cracked Pavement
  - UP: Uncracked Pavement

- W (Walls)
  - CW: Cracked Wall
  - UW: Uncracked Wall

For this project, we transform this multi-folder structure into a binary classification problem:

- Crack: CD, CP, CW
- No Crack: UD, UP, UW

This allows the deep learning models to learn crack patterns across multiple concrete surface types instead of focusing on only one surface category.

In [ ]:
# 2. Count Images in Each Category

from glob import glob

counts = {}

for key, path in paths.items():
    image_files = glob(os.path.join(path, "*.jpg"))
    counts[key] = len(image_files)

# Display counts
for k, v in counts.items():
    print(f"{k}: {v}")

# Total cracked vs uncracked
crack_total = counts["CD"] + counts["CP"] + counts["CW"]
no_crack_total = counts["UD"] + counts["UP"] + counts["UW"]

print("\nTotal Crack Images:", crack_total)
print("Total No Crack Images:", no_crack_total)

## Exploratory Data Analysis (EDA)

Before training deep learning models, it is important to analyze the dataset distribution to understand class balance and potential biases.

### Image Distribution by Category

- CD (Cracked Deck): 2,025  
- UD (Uncracked Deck): 11,595  

- CP (Cracked Pavement): 2,608  
- UP (Uncracked Pavement): 21,726  

- CW (Cracked Wall): 3,851  
- UW (Uncracked Wall): 14,287  

### Overall Class Distribution

- Total Crack Images: 8,484  
- Total No Crack Images: 47,608  

### Key Observations

- The dataset is **highly imbalanced**, with significantly more "No Crack" images than "Crack" images.
- This imbalance can bias the model toward predicting the majority class.
- Missing cracks (false negatives) is critical in real-world scenarios, so handling imbalance is important.
- The dataset includes multiple surface types (deck, pavement, wall), which improves generalization but also introduces variability.

### Next Step

We will visualize this imbalance and later apply techniques such as class weighting or data augmentation to mitigate its impact.

In [ ]:
# 3. Visualize Class Distribution

import matplotlib.pyplot as plt

labels = ["Crack", "No Crack"]
values = [crack_total, no_crack_total]

plt.figure(figsize=(6, 4))
plt.bar(labels, values)
plt.title("Class Distribution")
plt.ylabel("Number of Images")

for i, v in enumerate(values):
    plt.text(i, v + 500, str(v), ha='center')

plt.show()

### Class Imbalance Analysis

The class distribution shows a significant imbalance between cracked and uncracked images.

- Crack images: 8,484  
- No Crack images: 47,608  

This indicates that the dataset is heavily skewed toward the "No Crack" class.

#### Implications

- A model trained on this data may become biased toward predicting "No Crack".
- High accuracy alone may be misleading if the model ignores the minority class.
- In real-world scenarios, missing a crack (false negative) is critical and must be minimized.

#### Strategy to Handle Imbalance

To address this issue, we will:

- Use evaluation metrics such as recall and F1-score instead of relying only on accuracy
- Apply data augmentation to improve representation of crack images
- Consider class weighting during training

This ensures the model learns meaningful patterns for crack detection instead of favoring the majority class.

In [ ]:
# 4. Visualize Sample Images from Dataset

import random

def show_samples(paths, label_name, title):
    images = []
    for path in paths:
        files = glob(os.path.join(path, "*.jpg"))
        if files:
            images.extend(files)

    sample_images = random.sample(images, 5)

    plt.figure(figsize=(12, 3))
    for i, img_path in enumerate(sample_images):
        img = Image.open(img_path)
        plt.subplot(1, 5, i + 1)
        plt.imshow(img)
        plt.title(label_name)
        plt.axis("off")
    plt.suptitle(title)
    plt.show()


# Crack samples
show_samples(
    [paths["CD"], paths["CP"], paths["CW"]],
    "Crack",
    "Sample Crack Images"
)

# No Crack samples
show_samples(
    [paths["UD"], paths["UP"], paths["UW"]],
    "No Crack",
    "Sample No Crack Images"
)

## Data Preparation and Preprocessing

Before training deep learning models, the dataset must be prepared in a format suitable for PyTorch.

This includes:

- Converting images into tensors
- Resizing images to a fixed size (224 × 224)
- Normalizing pixel values
- Splitting the dataset into training, validation, and test sets

We will also apply data augmentation to improve generalization and help mitigate class imbalance.

In [ ]:
# 5. Create Image File DataFrame

data = []

# Cracked images
for category in ["CD", "CP", "CW"]:
    for img_path in glob(os.path.join(paths[category], "*.jpg")):
        data.append({
            "image_path": img_path,
            "label": 1,
            "label_name": "Crack",
            "surface_type": category[1]
        })

# Uncracked images
for category in ["UD", "UP", "UW"]:
    for img_path in glob(os.path.join(paths[category], "*.jpg")):
        data.append({
            "image_path": img_path,
            "label": 0,
            "label_name": "No Crack",
            "surface_type": category[1]
        })

df = pd.DataFrame(data)

print("Total images:", len(df))
df.head()

### Dataset Preparation Summary

The dataset has been successfully converted into a structured format suitable for deep learning.

- Total images: 56,092  
- Classes:
  - Crack (1)
  - No Crack (0)

Additional information retained:
- Surface type (Deck, Pavement, Wall) is preserved for potential analysis.

This structured DataFrame allows flexible data handling, including splitting, preprocessing, and loading into PyTorch pipelines.

Next, we will split the dataset into training, validation, and test sets.

In [ ]:
# 6. Split Dataset into Train, Validation, and Test Sets

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("Training set:", len(train_df))
print("Validation set:", len(val_df))
print("Test set:", len(test_df))

print("\nTraining class distribution:")
print(train_df["label_name"].value_counts())

print("\nValidation class distribution:")
print(val_df["label_name"].value_counts())

print("\nTest class distribution:")
print(test_df["label_name"].value_counts())

In [ ]:
# 3. Device Setup

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
# 7. Define Image Transforms

from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
# 8. Create Custom PyTorch Dataset

from torch.utils.data import Dataset

class CrackDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "image_path"]
        label = self.df.loc[idx, "label"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# 9. Create PyTorch Dataset Objects

train_dataset = CrackDataset(train_df, transform=train_transform)
val_dataset = CrackDataset(val_df, transform=val_test_transform)
test_dataset = CrackDataset(test_df, transform=val_test_transform)

print("Training dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

In [ ]:
# 10. Create DataLoaders

from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0  # important for Mac (MPS)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully.")

## Model Development

In this project, we use transfer learning with three powerful pretrained convolutional neural networks:

- ResNet50
- DenseNet121
- EfficientNet-B0

These models are pretrained on ImageNet and will be fine-tuned for binary classification (Crack vs No Crack).

### Approach

- Load pretrained weights
- Replace the final classification layer
- Train using the same dataset and parameters
- Compare performance using multiple evaluation metrics

We begin with ResNet50 as our first model.

In [ ]:
# 11. Build ResNet50 Model

from torchvision.models import resnet50, ResNet50_Weights

resnet_weights = ResNet50_Weights.DEFAULT
resnet_model = resnet50(weights=resnet_weights)

# Replace final classification layer for binary classification
num_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_features, 2)

resnet_model = resnet_model.to(device)

print("ResNet50 model loaded and modified successfully.")

In [ ]:
# 12. Define Loss Function and Optimizer for ResNet50

# Class weights help reduce bias toward the majority class
class_counts = train_df["label"].value_counts().sort_index().values
class_weights = len(train_df) / (2 * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    resnet_model.parameters(),
    lr=0.0001
)

print("Class weights:", class_weights)
print("Loss function and optimizer are ready.")

In [ ]:
# 13. Define Training Function

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=5):
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device).long()

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)

        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device).long()

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = correct / total

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Validation Loss: {avg_val_loss:.4f}")
        print(f"Validation Accuracy: {val_accuracy:.4f}")
        print()

    return model, history

In [ ]:
# 13. Define Training Function (with Progress Bar)

from tqdm import tqdm

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=5):
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):
        print(f"\nEpoch [{epoch+1}/{epochs}]")

        model.train()
        running_train_loss = 0.0

        loop = tqdm(train_loader, desc="Training", leave=True)

        for images, labels in loop:
            images = images.to(device)
            labels = labels.to(device).long()

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

            loop.set_postfix(loss=loss.item())

        avg_train_loss = running_train_loss / len(train_loader)

        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation", leave=False):
                images = images.to(device)
                labels = labels.to(device).long()

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = correct / total

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Validation Loss: {avg_val_loss:.4f}")
        print(f"Validation Accuracy: {val_accuracy:.4f}")

    return model, history

In [ ]:
# 14. Train ResNet50 Model

EPOCHS = 5

resnet_model, resnet_history = train_model(
    resnet_model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=EPOCHS
)

### ResNet50 Training Summary

The ResNet50 model was trained for 5 epochs using transfer learning and class-weighted loss to address the dataset imbalance.

The validation accuracy improved from 0.8731 in Epoch 1 to a peak of 0.9277 in Epoch 4. This indicates that the model learned meaningful crack-related visual patterns from the dataset.

However, validation accuracy slightly decreased in Epoch 5, suggesting that the model may be starting to fluctuate or mildly overfit. Therefore, final evaluation on the test set is required to better understand real-world generalization.

Since the dataset is imbalanced, accuracy alone is not enough. The next step is to evaluate the model using precision, recall, F1-score, and a confusion matrix.

In [ ]:
# 15. Evaluate ResNet50 on Test Set

all_preds = []
all_labels = []

resnet_model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).long()

        outputs = resnet_model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


# Convert to numpy
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)


# Metrics
acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print("Test Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))

### ResNet50 Test Evaluation Analysis

The ResNet50 model achieved the following performance on the test set:

- Accuracy: 0.9019  
- Precision (Crack): 0.6341  
- Recall (Crack): 0.8310  
- F1 Score: 0.7193  

### Class-wise Performance

- No Crack (Class 0):
  - Precision: 0.97
  - Recall: 0.91
  - F1 Score: 0.94

- Crack (Class 1):
  - Precision: 0.63
  - Recall: 0.83
  - F1 Score: 0.72

### Key Observations

- The model performs very well on the "No Crack" class, which is expected due to class imbalance.
- Recall for the "Crack" class is relatively high (0.83), which is important because it means most cracks are detected.
- Precision for the "Crack" class is lower (0.63), indicating some false positives (predicting crack when there is none).
- This trade-off is acceptable in real-world applications, where missing a crack is more critical than false alarms.

### Conclusion

The model demonstrates strong overall performance, with particularly good recall for crack detection. However, there is room for improvement in precision, which can be addressed in future model tuning or by comparing with other architectures such as DenseNet and EfficientNet.

In [ ]:
# 16. Plot Training and Validation Metrics

epochs = range(1, len(resnet_history["train_loss"]) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, resnet_history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs, resnet_history["val_loss"], marker="o", label="Validation Loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, resnet_history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("Validation Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(epochs)
plt.legend()

plt.show()

### Training Curve Analysis

The training and validation curves provide insight into the learning behavior of the ResNet50 model.

### Loss Curve

- Training loss shows a steady decrease across epochs, indicating that the model is learning effectively.
- Validation loss initially increases slightly, then decreases and stabilizes.
- A slight increase in validation loss in the final epoch suggests mild overfitting.

### Validation Accuracy Curve

- Validation accuracy improves significantly from Epoch 1 to Epoch 4.
- The highest accuracy is achieved at Epoch 4 (~92.7%).
- A small drop in Epoch 5 indicates that further training does not improve generalization.

### Key Insights

- The model converges well and learns meaningful patterns.
- There is no severe overfitting, but slight fluctuations suggest that optimal training occurs around Epoch 4.
- Early stopping around Epoch 4 could improve efficiency without sacrificing performance.

### Conclusion

The training process is stable, and the model demonstrates strong generalization. The results confirm that ResNet50 is a suitable architecture for crack detection, though further improvements can be explored using other models.

In [ ]:
# 17. Confusion Matrix Visualization

import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Crack", "Crack"],
            yticklabels=["No Crack", "Crack"])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()

### Confusion Matrix Analysis

The confusion matrix provides a detailed breakdown of the model's predictions:

- True Negatives (No Crack correctly predicted): 6532  
- False Positives (No Crack predicted as Crack): 610  
- False Negatives (Crack missed): 215  
- True Positives (Crack correctly detected): 1057  

### Key Insights

- The model performs very well in identifying "No Crack" images, as seen by the high number of true negatives.
- The number of false negatives (215) is relatively low, which is important because missing cracks is critical in real-world applications.
- The model has a moderate number of false positives (610), meaning it sometimes predicts cracks where none exist.

### Interpretation

- High true positive and low false negative values indicate that the model is effective at detecting cracks.
- The presence of false positives suggests the model is slightly conservative, preferring to flag potential cracks rather than miss them.

### Real-World Perspective

In structural inspection systems, this behavior is desirable:

- Missing a crack (false negative) can lead to safety risks
- Flagging extra cracks (false positives) only leads to additional inspection

Therefore, the model demonstrates a practical and safety-oriented prediction pattern.

## Model 2: DenseNet121

DenseNet121 is a deep convolutional neural network that connects each layer to every other layer in a feed-forward fashion.

### Key Idea

- Each layer receives inputs from all previous layers
- Improves feature reuse
- Reduces vanishing gradient problem

### Why DenseNet?

- Better at capturing fine-grained features (important for crack patterns)
- Often performs well on texture-heavy datasets
- Efficient parameter usage

We now implement DenseNet121 using transfer learning for crack classification.

In [ ]:
# 18. Build DenseNet121 Model

from torchvision.models import densenet121, DenseNet121_Weights

densenet_weights = DenseNet121_Weights.DEFAULT
densenet_model = densenet121(weights=densenet_weights)

# Replace final classification layer for binary classification
num_features = densenet_model.classifier.in_features
densenet_model.classifier = nn.Linear(num_features, 2)

densenet_model = densenet_model.to(device)

print("DenseNet121 model loaded and modified successfully.")

In [ ]:
# 19. Define Loss Function and Optimizer for DenseNet121

criterion_densenet = nn.CrossEntropyLoss(weight=class_weights)

optimizer_densenet = optim.Adam(
    densenet_model.parameters(),
    lr=0.0001
)

print("Loss function and optimizer for DenseNet121 are ready.")

In [ ]:
# 20. Train DenseNet121 Model

EPOCHS = 5

densenet_model, densenet_history = train_model(
    densenet_model,
    train_loader,
    val_loader,
    criterion_densenet,
    optimizer_densenet,
    epochs=EPOCHS
)

### DenseNet121 Training Summary

The DenseNet121 model was trained for 5 epochs using transfer learning and class-weighted loss to handle dataset imbalance.

Validation accuracy improved significantly from 0.8443 in Epoch 1 to a peak of 0.9334 in Epoch 3. After this point, performance slightly declined, indicating that the model reached its optimal generalization early in training.

### Key Observations

- DenseNet121 achieved its best validation accuracy earlier (Epoch 3) compared to ResNet50 (Epoch 4).
- Training loss steadily decreased, showing stable learning.
- Slight drop in validation accuracy after Epoch 3 suggests mild overfitting.

### Insight

DenseNet appears to learn faster and capture crack patterns efficiently due to its dense connectivity, which encourages feature reuse and better gradient flow.

Next, we visualize the training curves to analyze convergence behavior.

In [ ]:
# 21. Plot DenseNet121 Training and Validation Metrics

epochs = range(1, len(densenet_history["train_loss"]) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, densenet_history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs, densenet_history["val_loss"], marker="o", label="Validation Loss")
plt.title("DenseNet121 Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, densenet_history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("DenseNet121 Validation Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(epochs)
plt.legend()

plt.show()

### DenseNet121 Training Curve Analysis

The training curves for DenseNet121 show stable and efficient learning behavior.

### Loss Curve

- Training loss decreases steadily across epochs, indicating consistent learning.
- Validation loss also decreases and stabilizes without significant spikes.
- The gap between training and validation loss is small, suggesting good generalization.

### Validation Accuracy Curve

- Validation accuracy improves rapidly from Epoch 1 to Epoch 3.
- Peak performance is achieved at Epoch 3 (~93.3%).
- Slight decline after Epoch 3 indicates mild overfitting.

### Key Insights

- DenseNet121 converges faster than ResNet50, reaching optimal performance earlier.
- The model shows strong generalization with minimal overfitting.
- Early stopping around Epoch 3 could be optimal.

### Comparison Insight (Early)

- DenseNet achieves higher peak validation accuracy than ResNet.
- DenseNet learns faster and stabilizes earlier.

### Conclusion

DenseNet121 demonstrates strong performance for crack detection, with efficient learning and better convergence behavior compared to ResNet50.

In [ ]:
# 22. Evaluate DenseNet121 on Test Set

densenet_preds = []
densenet_labels = []

densenet_model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).long()

        outputs = densenet_model(images)
        _, preds = torch.max(outputs, 1)

        densenet_preds.extend(preds.cpu().numpy())
        densenet_labels.extend(labels.cpu().numpy())

densenet_preds = np.array(densenet_preds)
densenet_labels = np.array(densenet_labels)

densenet_acc = accuracy_score(densenet_labels, densenet_preds)
densenet_precision = precision_score(densenet_labels, densenet_preds)
densenet_recall = recall_score(densenet_labels, densenet_preds)
densenet_f1 = f1_score(densenet_labels, densenet_preds)

print("DenseNet121 Test Accuracy:", densenet_acc)
print("DenseNet121 Precision:", densenet_precision)
print("DenseNet121 Recall:", densenet_recall)
print("DenseNet121 F1 Score:", densenet_f1)

print("\nClassification Report:\n")
print(classification_report(densenet_labels, densenet_preds))

### DenseNet121 Test Evaluation Analysis

The DenseNet121 model achieved the following performance on the test set:

- Accuracy: 0.8965  
- Precision (Crack): 0.6128  
- Recall (Crack): 0.8561  
- F1 Score: 0.7143  

### Class-wise Performance

- No Crack (Class 0):
  - Precision: 0.97
  - Recall: 0.90
  - F1 Score: 0.94

- Crack (Class 1):
  - Precision: 0.61
  - Recall: 0.86
  - F1 Score: 0.71

### Key Observations

- The model maintains strong performance for the "No Crack" class.
- Recall for the "Crack" class (0.86) is slightly higher than ResNet50, meaning fewer missed cracks.
- Precision for the "Crack" class is slightly lower, indicating more false positives.
- Overall F1 score is similar to ResNet, suggesting comparable performance.

### Comparison with ResNet50

- DenseNet achieves higher recall (better at detecting cracks)
- ResNet achieves slightly higher precision (fewer false alarms)
- Both models have similar overall performance

### Conclusion

DenseNet121 is slightly more aggressive in detecting cracks, making it more suitable for safety-critical applications where missing a crack is more dangerous than raising false alarms.

In [ ]:
# 23. Confusion Matrix for DenseNet121

cm_dense = confusion_matrix(densenet_labels, densenet_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_dense, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Crack", "Crack"],
            yticklabels=["No Crack", "Crack"])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("DenseNet121 Confusion Matrix")

plt.show()

### DenseNet121 Confusion Matrix Analysis

The confusion matrix for DenseNet121 shows the following results:

- True Negatives (No Crack correctly predicted): 6454  
- False Positives (No Crack predicted as Crack): 688  
- False Negatives (Crack missed): 183  
- True Positives (Crack correctly detected): 1089  

### Key Observations

- DenseNet detects more cracks correctly (higher true positives) compared to ResNet50.
- The number of missed cracks (false negatives = 183) is lower than ResNet (215), which is a significant improvement.
- However, DenseNet produces more false positives (688 vs 610 in ResNet), meaning it flags more non-crack images as cracks.

### Comparison with ResNet50

- DenseNet has fewer missed cracks → better recall
- DenseNet has more false alarms → lower precision
- Overall, DenseNet is more aggressive in detecting cracks

### Real-World Interpretation

This behavior is often desirable in safety-critical systems:

- Missing cracks is dangerous → DenseNet performs better here
- Extra false alarms are acceptable → they only require additional inspection

### Conclusion

DenseNet121 demonstrates improved crack detection capability compared to ResNet50, especially in reducing false negatives, making it more suitable for real-world structural monitoring systems.

## Model 3: EfficientNet-B0

EfficientNet is a modern convolutional neural network architecture that scales model depth, width, and resolution in a balanced way.

### Key Idea

- Uses compound scaling to improve performance efficiently
- Achieves high accuracy with fewer parameters
- Optimized for both performance and computational efficiency

### Why EfficientNet?

- Strong performance on image classification tasks
- Efficient learning compared to traditional CNNs
- Widely used in real-world applications

We now implement EfficientNet-B0 using transfer learning for crack detection.

In [ ]:
# 24. Build EfficientNet-B0 Model

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

efficientnet_weights = EfficientNet_B0_Weights.DEFAULT
efficientnet_model = efficientnet_b0(weights=efficientnet_weights)

# Replace final classification layer for binary classification
num_features = efficientnet_model.classifier[1].in_features
efficientnet_model.classifier[1] = nn.Linear(num_features, 2)

efficientnet_model = efficientnet_model.to(device)

print("EfficientNet-B0 model loaded and modified successfully.")

In [ ]:
# 25. Define Loss Function and Optimizer for EfficientNet-B0

criterion_efficientnet = nn.CrossEntropyLoss(weight=class_weights)

optimizer_efficientnet = optim.Adam(
    efficientnet_model.parameters(),
    lr=0.0001
)

print("Loss function and optimizer for EfficientNet-B0 are ready.")

In [ ]:
# 26. Train EfficientNet-B0 Model

EPOCHS = 5

efficientnet_model, efficientnet_history = train_model(
    efficientnet_model,
    train_loader,
    val_loader,
    criterion_efficientnet,
    optimizer_efficientnet,
    epochs=EPOCHS
)

### EfficientNet-B0 Training Summary

The EfficientNet-B0 model was trained for 5 epochs using transfer learning.

### Key Observations

- Validation accuracy remained stable across epochs.
- Unlike DenseNet, EfficientNet continued improving and achieved its best performance at Epoch 5.
- Peak validation accuracy reached approximately 93.31%, which is slightly higher than both ResNet50 and DenseNet121.

### Insight

EfficientNet demonstrates:

- Stable learning behavior
- Better generalization across epochs
- Improved final performance compared to other models

### Conclusion

EfficientNet-B0 shows strong potential as the best-performing model so far, combining both stability and high accuracy in crack detection.

# 27. Plot EfficientNet-B0 Training and Validation Metrics

epochs = range(1, len(efficientnet_history["train_loss"]) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, efficientnet_history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs, efficientnet_history["val_loss"], marker="o", label="Validation Loss")
plt.title("EfficientNet-B0 Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, efficientnet_history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("EfficientNet-B0 Validation Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(epochs)
plt.legend()

plt.show()

In [ ]:
# 28. Evaluate EfficientNet-B0 on Test Set

eff_preds = []
eff_labels = []

efficientnet_model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).long()

        outputs = efficientnet_model(images)
        _, preds = torch.max(outputs, 1)

        eff_preds.extend(preds.cpu().numpy())
        eff_labels.extend(labels.cpu().numpy())

eff_preds = np.array(eff_preds)
eff_labels = np.array(eff_labels)

eff_acc = accuracy_score(eff_labels, eff_preds)
eff_precision = precision_score(eff_labels, eff_preds)
eff_recall = recall_score(eff_labels, eff_preds)
eff_f1 = f1_score(eff_labels, eff_preds)

print("EfficientNet Test Accuracy:", eff_acc)
print("Precision:", eff_precision)
print("Recall:", eff_recall)
print("F1 Score:", eff_f1)

print("\nClassification Report:\n")
print(classification_report(eff_labels, eff_preds))

In [ ]:
# 29. Confusion Matrix for EfficientNet-B0

cm_eff = confusion_matrix(eff_labels, eff_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_eff, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Crack", "Crack"],
            yticklabels=["No Crack", "Crack"])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("EfficientNet-B0 Confusion Matrix")

plt.show()

### EfficientNet-B0 Confusion Matrix Analysis

The confusion matrix for EfficientNet-B0 shows the following results:

- True Negatives (No Crack correctly predicted): 6745  
- False Positives (No Crack predicted as Crack): 397  
- False Negatives (Crack missed): 234  
- True Positives (Crack correctly detected): 1038  

### Observations

- The model correctly classifies a large number of "No Crack" images.
- The number of false positives (397) indicates instances where non-crack images are predicted as cracks.
- The model successfully detects a substantial number of cracks (1038 true positives).
- There are 234 instances where cracks are missed (false negatives).

These values provide a detailed view of how the model performs across both classes and highlight the types of prediction errors made.

In [ ]:
# 30. Recompute Metrics Safely + Plot Comparisons

# --- ResNet ---
resnet_acc = accuracy_score(all_labels, all_preds)
resnet_precision = precision_score(all_labels, all_preds)
resnet_recall = recall_score(all_labels, all_preds)
resnet_f1 = f1_score(all_labels, all_preds)

# --- DenseNet ---
dense_acc = accuracy_score(densenet_labels, densenet_preds)
dense_precision = precision_score(densenet_labels, densenet_preds)
dense_recall = recall_score(densenet_labels, densenet_preds)
dense_f1 = f1_score(densenet_labels, densenet_preds)

# --- EfficientNet ---
eff_acc = accuracy_score(eff_labels, eff_preds)
eff_precision = precision_score(eff_labels, eff_preds)
eff_recall = recall_score(eff_labels, eff_preds)
eff_f1 = f1_score(eff_labels, eff_preds)

models = ["ResNet50", "DenseNet121", "EfficientNet-B0"]

accuracy_vals = [resnet_acc, dense_acc, eff_acc]
precision_vals = [resnet_precision, dense_precision, eff_precision]
recall_vals = [resnet_recall, dense_recall, eff_recall]
f1_vals = [resnet_f1, dense_f1, eff_f1]

val_acc_vals = [
    max(resnet_history["val_accuracy"]),
    max(densenet_history["val_accuracy"]),
    max(efficientnet_history["val_accuracy"])
]

loss_vals = [
    resnet_history["val_loss"][-1],
    densenet_history["val_loss"][-1],
    efficientnet_history["val_loss"][-1]
]

# Plot function
def plot_metric(title, values, ylabel):
    plt.figure(figsize=(7, 4))
    bars = plt.bar(models, values)
    plt.title(title)
    plt.ylabel(ylabel)

    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, value + 0.01, f"{value:.3f}", ha='center')

    plt.show()

# Plot all
plot_metric("Test Accuracy", accuracy_vals, "Accuracy")
plot_metric("Precision", precision_vals, "Precision")
plot_metric("Recall", recall_vals, "Recall")
plot_metric("F1 Score", f1_vals, "F1 Score")
plot_metric("Best Validation Accuracy", val_acc_vals, "Val Accuracy")

plt.figure(figsize=(7, 4))
bars = plt.bar(models, loss_vals)
plt.title("Final Validation Loss")
plt.ylabel("Loss")

for bar, value in zip(bars, loss_vals):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.01, f"{value:.3f}", ha='center')

plt.show()

## Final Model Comparison and Selection

In this study, three deep learning models—ResNet50, DenseNet121, and EfficientNet-B0—were evaluated for the task of crack detection using multiple performance metrics, including accuracy, precision, recall, F1 score, validation accuracy, and loss.

### Accuracy

EfficientNet-B0 achieved the highest test accuracy (~0.925), indicating the best overall classification performance across both crack and non-crack classes.

### Precision

EfficientNet-B0 recorded the highest precision (~0.72), meaning it produces fewer false positives compared to the other models. This is important in real-world scenarios where false alarms can lead to unnecessary inspections and increased costs.

### Recall

DenseNet121 achieved the highest recall (~0.856), indicating it is more sensitive in detecting cracks and minimizes missed crack cases. However, this comes at the cost of increased false positives.

### F1 Score

EfficientNet-B0 achieved the highest F1 score (~0.767), demonstrating the best balance between precision and recall. Since F1 score considers both false positives and false negatives, it is a critical metric for evaluating overall model performance.

### Validation Accuracy and Loss

Both DenseNet121 and EfficientNet-B0 showed similar high validation accuracy (~0.933), indicating strong generalization capability. However, EfficientNet-B0 achieved the lowest validation loss (~0.270), suggesting more stable and efficient learning.

### Comparative Analysis

- **ResNet50**: Provides stable and consistent performance but does not outperform the other models in any specific metric.
- **DenseNet121**: Excels in recall, making it suitable for applications where detecting all possible cracks is critical, even at the expense of more false positives.
- **EfficientNet-B0**: Achieves the best balance across all metrics, including the highest accuracy, precision, and F1 score, along with low loss.

### Model Selection

Based on the overall comparison:

- EfficientNet-B0 is the most balanced and reliable model.
- It effectively minimizes both false positives and false negatives.
- It maintains strong generalization and stable learning behavior.

### Conclusion

EfficientNet-B0 can be considered the best-performing model for this crack detection task, as it provides the optimal trade-off between accuracy, precision, recall, and overall robustness.

### Additional Note

Although some models showed peak performance at earlier epochs during training, the final evaluation was conducted using the fully trained models after all epochs to ensure consistency across all comparisons.

## Hyperparameter Tuning for EfficientNet-B0

After comparing the performance of ResNet50, DenseNet121, and EfficientNet-B0, EfficientNet-B0 was selected as the best-performing model based on its superior accuracy, precision, and F1 score.

To further improve model performance, hyperparameter tuning is applied to EfficientNet-B0.

### Objective

The goal of hyperparameter tuning is to identify the optimal configuration that improves model performance, particularly in terms of:

- F1 Score (balance between precision and recall)
- Recall (to minimize missed cracks)
- Overall generalization

### Tuning Strategy

A grid-based search strategy is used to explore different combinations of key hyperparameters:

- Learning Rate
- Batch Size

Each configuration is trained for a fixed number of epochs, and performance is evaluated on the validation set.

The best configuration is selected based on validation performance and later evaluated on the test set.

In [ ]:
# 31. Define EfficientNet-B0 Builder Function for Tuning

def build_efficientnet_b0():
    weights = EfficientNet_B0_Weights.DEFAULT
    model = efficientnet_b0(weights=weights)

    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, 2)

    model = model.to(device)

    return model

In [ ]:
# 32. Define Validation Evaluation Function for Tuning

def evaluate_model_on_validation(model, val_loader):
    model.eval()

    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).long()

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_accuracy = accuracy_score(val_labels, val_preds)
    val_precision = precision_score(val_labels, val_preds)
    val_recall = recall_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds)

    return val_accuracy, val_precision, val_recall, val_f1

In [ ]:
# 33. Hyperparameter Tuning Configuration

learning_rates = [1e-4, 5e-5, 1e-5]
batch_sizes = [16, 32]

TUNING_EPOCHS = 4

tuning_results = []

print("Hyperparameter tuning setup is ready.")
print("Learning rates:", learning_rates)
print("Batch sizes:", batch_sizes)
print("Epochs per configuration:", TUNING_EPOCHS)

In [ ]:
# 34. Run Grid-Based Hyperparameter Tuning (with Detailed Runtime Output)

from torch.utils.data import DataLoader

total_configs = len(learning_rates) * len(batch_sizes)
config_num = 1

for lr in learning_rates:
    for batch_size in batch_sizes:

        print("\n" + "="*50)
        print(f"Config {config_num}/{total_configs}")
        print(f"Learning Rate: {lr} | Batch Size: {batch_size}")
        print("="*50)

        # Rebuild model
        model = build_efficientnet_b0()

        # DataLoaders
        train_loader_tune = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0
        )

        val_loader_tune = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )

        # Loss + optimizer
        criterion_tune = nn.CrossEntropyLoss(weight=class_weights)
        optimizer_tune = optim.Adam(model.parameters(), lr=lr)

        print("Starting training...")

        # Train model
        model, _ = train_model(
            model,
            train_loader_tune,
            val_loader_tune,
            criterion_tune,
            optimizer_tune,
            epochs=TUNING_EPOCHS
        )

        print("Evaluating on validation set...")

        # Evaluate
        val_acc, val_prec, val_rec, val_f1 = evaluate_model_on_validation(model, val_loader_tune)

        print(f"Results → Accuracy: {val_acc:.4f}, Precision: {val_prec:.4f}, Recall: {val_rec:.4f}, F1: {val_f1:.4f}")

        tuning_results.append({
            "learning_rate": lr,
            "batch_size": batch_size,
            "val_accuracy": val_acc,
            "val_precision": val_prec,
            "val_recall": val_rec,
            "val_f1": val_f1
        })

        config_num += 1

print("\n" + "="*50)
print("All configurations completed.")
print("="*50)

In [ ]:
# 35. Display Tuning Results and Insights

tuning_df = pd.DataFrame(tuning_results)

# Sort by F1 score (most important)
tuning_df_sorted = tuning_df.sort_values(by="val_f1", ascending=False)

print("\n=== ALL TUNING RESULTS (Sorted by F1 Score) ===\n")
print(tuning_df_sorted)

# Best configuration
best_config = tuning_df_sorted.iloc[0]

print("\n" + "="*50)
print("BEST CONFIGURATION FOUND")
print("="*50)

print(f"Learning Rate : {best_config['learning_rate']}")
print(f"Batch Size    : {best_config['batch_size']}")
print(f"Validation Accuracy : {best_config['val_accuracy']:.4f}")
print(f"Validation Precision: {best_config['val_precision']:.4f}")
print(f"Validation Recall   : {best_config['val_recall']:.4f}")
print(f"Validation F1 Score : {best_config['val_f1']:.4f}")

# Additional insights
print("\n=== INSIGHTS ===")
print("- Higher F1 score indicates better balance between precision and recall.")
print("- Recall is important to minimize missed cracks.")
print("- Precision helps reduce false alarms.")
print("- Best configuration balances all these metrics.")

### Hyperparameter Tuning Results

The hyperparameter tuning process improved the EfficientNet-B0 model performance.

The best configuration found was:

- Learning Rate: 0.00005
- Batch Size: 16

This configuration achieved:

- Validation Accuracy: 0.9355
- Validation Precision: 0.7668
- Validation Recall: 0.8240
- Validation F1 Score: 0.7944

Compared to the original EfficientNet-B0 model, the tuned configuration improved the F1 score and precision while maintaining strong recall. This suggests that the tuned model provides a better balance between detecting cracks and reducing false alarms.

In [ ]:
# 36. Train Final Tuned EfficientNet-B0 Model

BEST_LR = 5e-05
BEST_BATCH_SIZE = 16
FINAL_EPOCHS = 6

from torch.utils.data import DataLoader

# Build model
tuned_efficientnet = build_efficientnet_b0()

# DataLoaders
train_loader_final = DataLoader(
    train_dataset,
    batch_size=BEST_BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader_final = DataLoader(
    val_dataset,
    batch_size=BEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# Loss + optimizer
criterion_final = nn.CrossEntropyLoss(weight=class_weights)
optimizer_final = optim.Adam(tuned_efficientnet.parameters(), lr=BEST_LR)

print("Starting final training with tuned hyperparameters...")

tuned_efficientnet, tuned_history = train_model(
    tuned_efficientnet,
    train_loader_final,
    val_loader_final,
    criterion_final,
    optimizer_final,
    epochs=FINAL_EPOCHS
)

In [ ]:
# 37. Evaluate Tuned EfficientNet-B0 on Test Set

tuned_preds = []
tuned_labels = []

tuned_efficientnet.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).long()

        outputs = tuned_efficientnet(images)
        _, preds = torch.max(outputs, 1)

        tuned_preds.extend(preds.cpu().numpy())
        tuned_labels.extend(labels.cpu().numpy())

tuned_preds = np.array(tuned_preds)
tuned_labels = np.array(tuned_labels)

tuned_acc = accuracy_score(tuned_labels, tuned_preds)
tuned_precision = precision_score(tuned_labels, tuned_preds)
tuned_recall = recall_score(tuned_labels, tuned_preds)
tuned_f1 = f1_score(tuned_labels, tuned_preds)

print("Tuned EfficientNet-B0 Test Accuracy:", tuned_acc)
print("Tuned EfficientNet-B0 Precision:", tuned_precision)
print("Tuned EfficientNet-B0 Recall:", tuned_recall)
print("Tuned EfficientNet-B0 F1 Score:", tuned_f1)

print("\nClassification Report:\n")
print(classification_report(tuned_labels, tuned_preds))

## Analysis of Tuned EfficientNet-B0 Results

The tuned EfficientNet-B0 model was evaluated on the test dataset to assess the impact of hyperparameter tuning.

### Observed Performance

* Test Accuracy: 0.9166
* Precision: 0.6797
* Recall: 0.8475
* F1 Score: 0.7544

### Key Observations

Compared to the original EfficientNet-B0 model:

* **Recall increased**, indicating improved sensitivity in detecting crack instances
* **Precision decreased**, leading to a higher number of false positives
* **F1 Score decreased**, reflecting a reduction in overall balance between precision and recall
* **Accuracy slightly decreased**, indicating no overall improvement in general performance

### Interpretation

The results show that hyperparameter tuning did not lead to a uniform improvement in model performance. Instead, it altered the model's behavior by increasing its ability to detect cracks (recall) while reducing prediction reliability (precision).

This demonstrates that tuning can shift the balance between evaluation metrics rather than consistently improving all of them.


In [ ]:
# 38. Manual ROC-AUC Style Curves Using Notebook Output Values Only

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import auc

# --------------------------------------------------
# Support values from notebook classification reports
# --------------------------------------------------
negative_support = 7142   # Class 0 = No Crack
positive_support = 1272   # Class 1 = Crack

# --------------------------------------------------
# Notebook output metrics only
# --------------------------------------------------
model_outputs = {
    "ResNet50": {
        "accuracy": 0.9019491323983837,
        "precision": 0.6340731853629274,
        "recall": 0.8309748427672956,
        "f1": 0.7192922762844505
    },
    "DenseNet121": {
        "accuracy": 0.8964820537199905,
        "precision": 0.6128306133933596,
        "recall": 0.8561320754716981,
        "f1": 0.7143325680551
    },
    "EfficientNet-B0": {
        "accuracy": 0.9250059424768243,
        "precision": 0.7233449477351916,
        "recall": 0.8160377358490566,
        "f1": 0.7669006280014776
    },
    "Tuned EfficientNet-B0": {
        "accuracy": 0.916567625386261,
        "precision": 0.6796973518284993,
        "recall": 0.8474842767295597,
        "f1": 0.7543736878936319
    }
}

# --------------------------------------------------
# Build ROC-style points from notebook metrics
# --------------------------------------------------
roc_results = {}

for model_name, metrics in model_outputs.items():
    precision = metrics["precision"]
    recall = metrics["recall"]

    # Reconstruct confusion matrix values from notebook outputs
    TP = round(recall * positive_support)
    FN = positive_support - TP

    FP = round((TP / precision) - TP)
    TN = negative_support - FP

    # ROC point
    TPR = TP / (TP + FN)          # Same as recall
    FPR = FP / (FP + TN)

    # Manual ROC-style curve has 3 points:
    # (0,0), actual model point, (1,1)
    fpr_points = np.array([0, FPR, 1])
    tpr_points = np.array([0, TPR, 1])

    manual_auc = auc(fpr_points, tpr_points)

    roc_results[model_name] = {
        "TP": TP,
        "FP": FP,
        "TN": TN,
        "FN": FN,
        "FPR": FPR,
        "TPR": TPR,
        "AUC": manual_auc
    }

# --------------------------------------------------
# Plot manual ROC-style curves
# --------------------------------------------------
plt.figure(figsize=(10, 7))

for model_name, values in roc_results.items():
    fpr_points = [0, values["FPR"], 1]
    tpr_points = [0, values["TPR"], 1]

    plt.plot(
        fpr_points,
        tpr_points,
        marker="o",
        linewidth=2,
        label=f"{model_name} AUC = {values['AUC']:.4f}"
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Manual ROC-AUC Style Curve Comparison Using Notebook Output Values")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

# --------------------------------------------------
# Print reconstructed values clearly
# --------------------------------------------------
print("=== Manual ROC-AUC Style Results Using Notebook Output Values Only ===\n")

for model_name, values in roc_results.items():
    print(model_name)
    print(f"TP: {values['TP']}")
    print(f"FP: {values['FP']}")
    print(f"TN: {values['TN']}")
    print(f"FN: {values['FN']}")
    print(f"TPR / Recall: {values['TPR']:.4f}")
    print(f"FPR: {values['FPR']:.4f}")
    print(f"Manual AUC: {values['AUC']:.4f}")
    print("-" * 50)

## Manual ROC-AUC Style Analysis

Because the notebook output provides final classification metrics but does not store probability scores, this ROC-AUC visualization was created manually using only the notebook’s printed precision, recall, and class support values.

This is not a full probability-based ROC curve. Instead, it represents each model using its observed classification operating point: false positive rate and true positive rate. The curve connects three points: the origin, the model’s observed performance point, and the perfect upper endpoint.

Based on this notebook-output-only comparison, the tuned EfficientNet-B0 shows the strongest manual ROC-style AUC because it achieves high recall while keeping the false positive rate relatively controlled. EfficientNet-B0 still remains the strongest model in terms of overall test accuracy and F1-score, while the tuned model improves crack detection sensitivity.

## Answer to Research Question

The research question for this project was:

**Can deep learning models accurately detect surface cracks in structural images while maintaining a balance between precision and recall?**

### Answer

The results of this study demonstrate that deep learning models are capable of accurately detecting surface cracks with strong performance across multiple evaluation metrics.

Among the models evaluated, EfficientNet-B0 achieved the best overall results, showing high accuracy and a well-balanced trade-off between precision and recall. This indicates that deep learning architectures can effectively learn both global and fine-grained features necessary for identifying crack patterns in images.

However, the analysis also revealed that maintaining a balance between precision and recall is not guaranteed and depends on model configuration. Hyperparameter tuning showed that improving recall (detecting more cracks) can lead to a reduction in precision (increased false positives), highlighting an inherent trade-off.

### Final Insight

Therefore, while deep learning models can successfully detect surface cracks with high accuracy, achieving an optimal balance between precision and recall requires careful model selection and tuning based on the specific requirements of the application.

This confirms that deep learning is a viable and effective solution for automated crack detection, but performance must be evaluated in the context of real-world priorities.


## Real-World Comparison and Implications

The performance of the developed models can be better understood when evaluated in the context of real-world structural inspection systems.

In practical applications such as bridge monitoring, building inspection, and road safety assessment, the consequences of prediction errors are not equal:

* **False Negatives (missed cracks)** can lead to undetected structural damage, posing serious safety risks
* **False Positives (incorrect crack detection)** may result in unnecessary inspections, increasing operational costs

---

### Model Behavior in Real-World Context

Based on the results:

* The **original EfficientNet-B0 model** provides a strong balance between precision and recall, making it suitable for general-purpose deployment where both accuracy and reliability are important

* The **tuned EfficientNet-B0 model** demonstrates higher recall, meaning it detects more cracks but also produces more false positives

---

### Application-Based Comparison

Different real-world scenarios may require different model behaviors:

* **Safety-Critical Systems (e.g., bridges, critical infrastructure)**
  → Priority: High Recall
  → Preferred Model: Tuned EfficientNet-B0
  → Reason: Missing a crack could lead to severe consequences

* **Cost-Sensitive Systems (e.g., routine inspections, large-scale monitoring)**
  → Priority: High Precision
  → Preferred Model: Original EfficientNet-B0
  → Reason: Reducing false alarms minimizes unnecessary inspections

---

### Key Insight

This comparison highlights that model selection should not be based solely on a single metric such as accuracy. Instead, it must align with the operational priorities and risk tolerance of the application.

---

### Final Takeaway

Deep learning models provide flexible solutions for crack detection, where performance can be adjusted based on real-world needs. Understanding and managing the trade-off between precision and recall is essential for deploying reliable and effective systems.


## Future Work

While this project demonstrates the effectiveness of deep learning models for surface crack detection, several opportunities exist to further improve performance and extend real-world applicability.

### 1. Advanced Hyperparameter Optimization

The current tuning approach used a controlled grid search. Future work can explore more advanced optimization techniques such as:

* Bayesian Optimization
* Optuna-based tuning
* Adaptive learning rate scheduling

These methods can more efficiently identify optimal configurations and potentially improve model performance without extensive manual experimentation.

---

### 2. Data Augmentation and Class Balancing

Given the class imbalance in the dataset, additional techniques can be applied:

* Image augmentation (rotation, brightness variation, noise injection)
* Synthetic data generation
* Oversampling of minority class

These approaches can improve the model’s ability to generalize and better detect crack instances.

---

### 3. Early Stopping and Regularization

The training results indicated that model performance peaked before the final epoch. Future improvements can include:

* Early stopping based on validation performance
* Dropout layers to reduce overfitting
* Weight regularization techniques

This would help retain the best-performing model state and improve generalization.

---

### 4. Exploration of Advanced Architectures

Further improvements may be achieved by experimenting with:

* Vision Transformers (ViT)
* Hybrid CNN-Transformer models
* Lightweight architectures for real-time deployment

These models may provide improved feature extraction and scalability.

---

### 5. Real-Time Deployment and Edge Integration

To make the system practical for real-world use, future work can focus on:

* Deploying the model on edge devices (e.g., Jetson platforms)
* Building real-time monitoring dashboards
* Integrating alert systems for detected cracks

This would enable continuous and automated infrastructure monitoring.

---

### 6. Multi-Class and Severity-Based Detection

Instead of binary classification, future systems can be extended to:

* Classify different types of cracks
* Detect crack severity levels
* Perform localization using object detection models

This would provide more actionable insights for maintenance and repair.

---

### Final Perspective

Future work should focus not only on improving model accuracy but also on enhancing interpretability, robustness, and deployment readiness. This will ensure that the system can transition from a research prototype to a reliable real-world solution.


In [ ]:
#hh

## Final Conclusion

This project explored the application of deep learning techniques for automated surface crack detection using structural image data. Three advanced convolutional neural network architectures—ResNet50, DenseNet121, and EfficientNet-B0—were implemented and evaluated to identify the most effective model.

The experimental results demonstrate that deep learning models are capable of accurately detecting surface cracks with strong performance across multiple evaluation metrics. Among the models tested, EfficientNet-B0 consistently achieved the best overall performance, providing a balanced combination of accuracy, precision, recall, and F1 score.

Further experimentation with hyperparameter tuning revealed an important insight: model optimization does not always lead to uniform improvement. Instead, tuning introduced a trade-off by increasing recall while reducing precision and overall F1 score. This highlights that improving one aspect of performance may come at the cost of another.

From a practical perspective, this emphasizes the importance of aligning model selection with real-world requirements:

* When minimizing missed cracks is critical, a recall-focused model is preferable
* When reducing false alarms is important, a precision-focused model is more suitable
* For general applications, a balanced model such as the original EfficientNet-B0 is the most appropriate choice

Overall, this study confirms that deep learning provides an effective and scalable solution for crack detection. More importantly, it demonstrates that model evaluation must go beyond accuracy and consider the trade-offs between different performance metrics to ensure reliable deployment in real-world scenarios.
